In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import cv2


from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

In [3]:
from pathlib import Path

# -----------------------------
# CLEAN DATASET
# -----------------------------

kodak_clean_dir = Path(
    "/kaggle/input/datasets/sherylmehta/kodak-dataset"
)

bsd500_clean_dir = Path(
    "/kaggle/input/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500/images"
)

div2k_clean_base_dir = Path(
    "/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images"
)


# -----------------------------
# NOISY BERNOULLI DATASET
# -----------------------------
# Modifica i nomi delle cartelle se nel tuo notebook Bernoulli
# hai salvato gli output con nomi leggermente diversi.

bernoulli_base_dir = Path(
    "/kaggle/input/notebooks/antoiann/bernoullinoise"
)

kodak_noisy_dir = bernoulli_base_dir / "kodak_bernoulli_p050"

bsd500_noisy_dir = bernoulli_base_dir / "bsd500_bernoulli_p050"

div2k_noisy_base_dir = bernoulli_base_dir / "div2k_bernoulli_p050"


# -----------------------------
# OUTPUT BASELINES
# -----------------------------

output_base_dir = Path(
    "/kaggle/working/bernoulli_p050_classical_baselines"
)

output_base_dir.mkdir(parents=True, exist_ok=True)

In [4]:
# -----------------------------
# BSD500 split
# -----------------------------

bsd500_train_clean_dir = bsd500_clean_dir / "train"
bsd500_val_clean_dir   = bsd500_clean_dir / "val"
bsd500_test_clean_dir  = bsd500_clean_dir / "test"

bsd500_train_noisy_dir = bsd500_noisy_dir / "train"
bsd500_val_noisy_dir   = bsd500_noisy_dir / "val"
bsd500_test_noisy_dir  = bsd500_noisy_dir / "test"


# -----------------------------
# DIV2K split
# -----------------------------

div2k_train_clean_dir = div2k_clean_base_dir / "DIV2K_train_HR" / "DIV2K_train_HR"
div2k_valid_clean_dir = div2k_clean_base_dir / "DIV2K_valid_HR" / "DIV2K_valid_HR"

div2k_train_noisy_dir = div2k_noisy_base_dir / "DIV2K_train_HR" / "DIV2K_train_HR"
div2k_valid_noisy_dir = div2k_noisy_base_dir / "DIV2K_valid_HR" / "DIV2K_valid_HR"

In [5]:
def apply_mean_filter(noisy_np, kernel_size=3):
    """
    Applica un filtro media a un'immagine RGB float in [0,1].
    """
    # Conversione da float [0,1] a uint8 [0,255]
    noisy_uint8 = (np.clip(noisy_np, 0, 1) * 255).astype(np.uint8)
    # Applicazione del filtro media tramite OpenCV
    filtered_uint8 = cv2.blur(
        noisy_uint8,
        (kernel_size, kernel_size)
    )

    filtered_np = filtered_uint8.astype(np.float32) / 255.0

    return filtered_np

In [6]:
def apply_median_filter(noisy_np, kernel_size=3):
    """
    Applica un filtro mediano a un'immagine RGB float in [0,1].
    """

    noisy_uint8 = (np.clip(noisy_np, 0, 1) * 255).astype(np.uint8)

    filtered_uint8 = cv2.medianBlur(
        noisy_uint8,
        kernel_size
    )

    filtered_np = filtered_uint8.astype(np.float32) / 255.0

    return filtered_np

In [7]:
def evaluate_baseline_dataset(
    method_name,
    filter_function,
    noisy_dir,
    clean_dir,
    output_dir,
    csv_name="metrics.csv",
    max_images=None
):
    """
    Applica un metodo di denoising classico a tutte le immagini rumorose
    contenute in noisy_dir, salva gli output e calcola PSNR/SSIM rispetto
    alle immagini clean corrispondenti.
    """

    noisy_dir = Path(noisy_dir)
    clean_dir = Path(clean_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    noisy_paths = get_image_paths(noisy_dir)

    if max_images is not None:
        noisy_paths = noisy_paths[:max_images]

    results = []

    for noisy_path in tqdm(noisy_paths, desc=method_name):
        try:
            relative_path = noisy_path.relative_to(noisy_dir)
            clean_path = clean_dir / relative_path

            if not clean_path.exists():
                print("Clean non trovata:", clean_path)
                continue

            # Lettura noisy
            noisy_img = Image.open(noisy_path).convert("RGB")
            noisy_np = np.array(noisy_img).astype(np.float32) / 255.0

            # Lettura clean
            clean_img = Image.open(clean_path).convert("RGB")
            clean_img = clean_img.resize((noisy_np.shape[1], noisy_np.shape[0]))
            clean_np = np.array(clean_img).astype(np.float32) / 255.0

            # Applicazione filtro
            denoised_np = filter_function(noisy_np)
            denoised_np = np.clip(denoised_np, 0, 1)

            # Salvataggio immagine denoised
            save_path = output_dir / relative_path
            save_path = save_path.with_suffix(".png")
            save_path.parent.mkdir(parents=True, exist_ok=True)

            Image.fromarray(
                (denoised_np * 255).astype(np.uint8)
            ).save(save_path)

            # Metriche noisy
            psnr_noisy = psnr(clean_np, noisy_np, data_range=1.0)
            ssim_noisy = ssim(clean_np, noisy_np, channel_axis=2, data_range=1.0)

            # Metriche metodo
            psnr_denoised = psnr(clean_np, denoised_np, data_range=1.0)
            ssim_denoised = ssim(clean_np, denoised_np, channel_axis=2, data_range=1.0)

            results.append({
                "method": method_name,
                "filename": str(relative_path),
                "clean_path": str(clean_path),
                "noisy_path": str(noisy_path),
                "denoised_path": str(save_path),
                "psnr_clean_noisy": float(psnr_noisy),
                "ssim_clean_noisy": float(ssim_noisy),
                "psnr_clean_denoised": float(psnr_denoised),
                "ssim_clean_denoised": float(ssim_denoised),
                "psnr_gain": float(psnr_denoised - psnr_noisy),
                "ssim_gain": float(ssim_denoised - ssim_noisy),
            })

        except Exception as e:
            print("Errore su:", noisy_path)
            print(e)

    df = pd.DataFrame(results)

    csv_path = output_dir / csv_name
    df.to_csv(csv_path, index=False)

    print("\nMetodo:", method_name)
    print("CSV salvato in:", csv_path)

    if len(df) > 0:
        print("PSNR medio clean/noisy:", df["psnr_clean_noisy"].mean())
        print("PSNR medio clean/denoised:", df["psnr_clean_denoised"].mean())
        print("Gain PSNR medio:", df["psnr_gain"].mean())

        print("SSIM medio clean/noisy:", df["ssim_clean_noisy"].mean())
        print("SSIM medio clean/denoised:", df["ssim_clean_denoised"].mean())
        print("Gain SSIM medio:", df["ssim_gain"].mean())

    return df

In [8]:
def get_image_paths(input_dir):
    """
    Restituisce tutti i path delle immagini contenute in input_dir,
    includendo eventuali sottocartelle.
    """

    input_dir = Path(input_dir)

    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]

    image_paths = [
        p for p in input_dir.rglob("*")
        if p.suffix.lower() in image_extensions
    ]

    return sorted(image_paths)

In [9]:
df_kodak_mean3 = evaluate_baseline_dataset(
    method_name="mean_3x3",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=3),
    noisy_dir=kodak_noisy_dir,
    clean_dir=kodak_clean_dir,
    output_dir=output_base_dir / "kodak_p050" / "mean_3x3",
    csv_name="kodak_bernoulli_p050_mean_3x3_metrics.csv"
)

mean_3x3: 100%|██████████| 24/24 [00:08<00:00,  2.78it/s]


Metodo: mean_3x3
CSV salvato in: /kaggle/working/bernoulli_p050_classical_baselines/kodak_p050/mean_3x3/kodak_bernoulli_p050_mean_3x3_metrics.csv
PSNR medio clean/noisy: 9.78579283275074
PSNR medio clean/denoised: 12.21465740817346
Gain PSNR medio: 2.4288645754227205
SSIM medio clean/noisy: 0.10899724485352635
SSIM medio clean/denoised: 0.26446513769527275
Gain SSIM medio: 0.15546789268652597


In [10]:
df_kodak_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=kodak_noisy_dir,
    clean_dir=kodak_clean_dir,
    output_dir=output_base_dir / "kodak_p050" / "median_3x3",
    csv_name="kodak_bernoulli_p050_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 24/24 [00:07<00:00,  3.32it/s]


Metodo: median_3x3
CSV salvato in: /kaggle/working/bernoulli_p050_classical_baselines/kodak_p050/median_3x3/kodak_bernoulli_p050_median_3x3_metrics.csv
PSNR medio clean/noisy: 9.78579283275074
PSNR medio clean/denoised: 9.67337991905451
Gain PSNR medio: -0.11241291369623092
SSIM medio clean/noisy: 0.10899724485352635
SSIM medio clean/denoised: 0.08391353200810651
Gain SSIM medio: -0.025083712845419843


In [11]:
df_bsd500_mean3 = evaluate_baseline_dataset(
    method_name="mean_3x3",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=3),
    noisy_dir=bsd500_test_noisy_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=output_base_dir / "bsd500_test_p050" / "mean_3x3",
    csv_name="bsd500_test_bernoulli_p050_mean_3x3_metrics.csv"
)

mean_3x3: 100%|██████████| 200/200 [00:20<00:00,  9.66it/s]


Metodo: mean_3x3
CSV salvato in: /kaggle/working/bernoulli_p050_classical_baselines/bsd500_test_p050/mean_3x3/bsd500_test_bernoulli_p050_mean_3x3_metrics.csv
PSNR medio clean/noisy: 10.063952174915498
PSNR medio clean/denoised: 12.446883127017689
Gain PSNR medio: 2.382930952102191
SSIM medio clean/noisy: 0.14081985112279655
SSIM medio clean/denoised: 0.3070741228759289
Gain SSIM medio: 0.16625427156686784


In [12]:
df_bsd500_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=bsd500_test_noisy_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=output_base_dir / "bsd500_test_p050" / "median_3x3",
    csv_name="bsd500_test_bernoulli_p050_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 200/200 [00:20<00:00,  9.70it/s]


Metodo: median_3x3
CSV salvato in: /kaggle/working/bernoulli_p050_classical_baselines/bsd500_test_p050/median_3x3/bsd500_test_bernoulli_p050_median_3x3_metrics.csv
PSNR medio clean/noisy: 10.063952174915498
PSNR medio clean/denoised: 9.995035902105332
Gain PSNR medio: -0.06891627281016494
SSIM medio clean/noisy: 0.14081985112279655
SSIM medio clean/denoised: 0.1133223121613264
Gain SSIM medio: -0.027497538961470128


In [13]:
df_div2k_mean3 = evaluate_baseline_dataset(
    method_name="mean_3x3",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=3),
    noisy_dir=div2k_valid_noisy_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=output_base_dir / "div2k_valid_p050" / "mean_3x3",
    csv_name="div2k_valid_bernoulli_p050_mean_3x3_metrics.csv"
)

mean_3x3: 100%|██████████| 100/100 [03:16<00:00,  1.97s/it]


Metodo: mean_3x3
CSV salvato in: /kaggle/working/bernoulli_p050_classical_baselines/div2k_valid_p050/mean_3x3/div2k_valid_bernoulli_p050_mean_3x3_metrics.csv
PSNR medio clean/noisy: 9.589380527180035
PSNR medio clean/denoised: 12.020069098508364
Gain PSNR medio: 2.4306885713283304
SSIM medio clean/noisy: 0.1633033352345228
SSIM medio clean/denoised: 0.31502766601741317
Gain SSIM medio: 0.15172433093190194


In [14]:
df_div2k_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=div2k_valid_noisy_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=output_base_dir / "div2k_valid_p050" / "median_3x3",
    csv_name="div2k_valid_bernoulli_p050_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 100/100 [02:58<00:00,  1.79s/it]


Metodo: median_3x3
CSV salvato in: /kaggle/working/bernoulli_p050_classical_baselines/div2k_valid_p050/median_3x3/div2k_valid_bernoulli_p050_median_3x3_metrics.csv
PSNR medio clean/noisy: 9.589380527180035
PSNR medio clean/denoised: 9.474004567523926
Gain PSNR medio: -0.11537595965610908
SSIM medio clean/noisy: 0.1633033352345228
SSIM medio clean/denoised: 0.13032650018110872
Gain SSIM medio: -0.03297683505341411
